In [1]:
# 03_figures_tables.ipynb
# Generates all figures (grayscale seaborn, 600 dpi, PNG + PDF, no captions) and result tables
# for the unified HITL-AI study. Figures are saved to ../results/figures and tables to ../results/tables.

## Setup - grayscale theme and I/O helpers

In [2]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

BASE = ".."
FIG = os.path.join(BASE, "results", "figures")
TAB = os.path.join(BASE, "results", "tables")
DATA = os.path.join(BASE, "data")
os.makedirs(FIG, exist_ok=True)
os.makedirs(TAB, exist_ok=True)

# Grayscale seaborn theme
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "font.size": 11,
    "axes.edgecolor": "0.2",
    "axes.linewidth": 0.8,
    "grid.color": "0.85",
})
GRAY = plt.cm.gray

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

df = pd.read_csv(os.path.join(DATA, "tasks.csv"))
df["g_i"] = df["pre_ai_hours"] - df["post_ai_hours"]
df["net_i"] = df["g_i"] - df["verification_hours"]
df["becker"] = df["net_i"] < 0

## Figure 1 - Delegation frontier map (g vs v)

In [3]:
# ---- Figure 1: Delegation frontier map (g vs v), Becker region shaded ----
fig, ax = plt.subplots(figsize=(6, 5))
gmax = 3.2
xs = np.linspace(-1.5, gmax, 200)
# Becker region: g < v  => below diagonal v = g ; shade region where v > g
ax.fill_between(xs, xs, 3.0, where=(xs < 3.0), color="0.75", alpha=0.6, zorder=0)
ax.plot([-1.5, 3.0], [-1.5, 3.0], color="0.1", lw=1.2, ls="--", zorder=1)
markers = {"high_compression": "o", "partial": "s", "partial_or_irreducible": "^", "irreducible": "D"}
# Per-task label offsets (points) to avoid overlap with markers and the region text.
label_off = {"T1": (8, -14), "T2": (8, 4), "T3": (10, -2), "T4": (8, 6),
             "T5": (8, 6), "T6": (10, 2), "T7": (-4, -16)}
for _, r in df.iterrows():
    ax.scatter(r["g_i"], r["verification_hours"],
               s=90, c="0.25", marker=markers.get(r["compression_type"], "o"),
               edgecolor="black", linewidth=0.7, zorder=3)
    dx, dy = label_off.get(r["task_id"], (6, 4))
    ax.annotate(r["task_id"], (r["g_i"], r["verification_hours"]),
                textcoords="offset points", xytext=(dx, dy), fontsize=9, color="0.1")
ax.set_xlabel("Time saving  $g_i$  (hours)")
ax.set_ylabel("Verification cost  $v_i$  (hours)")
# Region label placed in an empty part of the shaded area (upper-middle), clear of markers.
ax.text(0.30, 2.62, "Delegation-loss region ($v_i > g_i$)",
        fontsize=9, color="0.15", ha="left", va="center")
ax.set_xlim(-1.5, gmax)
ax.set_ylim(-0.2, 3.0)
savefig(fig, "fig1_delegation_frontier")
print("fig1 done")

fig1 done


## Figure 2 - Per-task net effect of delegation

In [4]:
# ---- Figure 2: Per-task net effect of delegation (bar) ----
fig, ax = plt.subplots(figsize=(6.5, 4))
order = df.sort_values("net_i")
colors = ["0.35" if n >= 0 else "0.7" for n in order["net_i"]]
ax.bar(order["task_id"], order["net_i"], color=colors, edgecolor="black", linewidth=0.7)
ax.axhline(0, color="0.1", lw=1.0)
ax.set_ylabel("Net time saving  $g_i - v_i$  (hours)")
ax.set_xlabel("Task")
for i, (tid, n) in enumerate(zip(order["task_id"], order["net_i"])):
    ax.annotate(f"{n:.1f}", (i, n), textcoords="offset points",
                xytext=(0, 3 if n >= 0 else -12), ha="center", fontsize=8, color="0.1")
savefig(fig, "fig2_net_effect")
print("fig2 done")

fig2 done


## Figure 3 - Governance-capacity tradeoff

In [5]:
# ---- Figure 3: Governance-capacity tradeoff (dc*/df = lambda*k) ----
fig, ax = plt.subplots(figsize=(6, 4.5))
lam = 45 / 40.0  # jobs/hour (working-hours basis)
E0 = 2.0
for k, ls in zip([0.5, 1.0, 1.5], ["-", "--", ":"]):
    f = np.linspace(0, 4, 100)
    c_star = lam * (E0 + k * f)
    ax.plot(f, c_star, color="0.2", ls=ls, lw=1.4, label=f"k = {k}")
ax.set_xlabel("Forced-friction intensity  $f$")
ax.set_ylabel("Minimum reviewers  $c^*(f) = \\lambda\\,E[S(f)]$")
ax.legend(title="Service-time\nsensitivity", frameon=True, edgecolor="0.5")
savefig(fig, "fig3_governance_capacity")
print("fig3 done")

fig3 done


## Figure 4 - Optimal-friction surface

In [6]:
# ---- Figure 4: Optimal friction f* surface (heatmap, grayscale) ----
fig, ax = plt.subplots(figsize=(6, 5))
cH = 1.0
E_err_grid = np.linspace(1, 30, 120)
k_grid = np.linspace(0.2, 2.0, 120)
EE, KK = np.meshgrid(E_err_grid, k_grid)
f_star = np.sqrt(EE / (cH * lam * KK)) - 1.0
f_star = np.clip(f_star, 0, None)
hm = ax.imshow(f_star, origin="lower", aspect="auto", cmap="gray_r",
               extent=[E_err_grid.min(), E_err_grid.max(), k_grid.min(), k_grid.max()])
cs = ax.contour(EE, KK, f_star, levels=[0.5, 1, 2, 3], colors="0.1", linewidths=0.8)
ax.clabel(cs, inline=True, fontsize=8, fmt="%.1f")
cbar = fig.colorbar(hm, ax=ax)
cbar.set_label("Optimal friction  $f^*$")
ax.set_xlabel("Error-cost scale  $E_{err}$")
ax.set_ylabel("Service-time sensitivity  $k$")
savefig(fig, "fig4_optimal_friction")
print("fig4 done")

fig4 done


## Figure 5 - M/G/c waiting time (Stage-6 bottleneck)

In [7]:
# ---- Figure 5: M/G/c waiting time vs arrival (KLB), Stage-6 bottleneck ----
def mmc_wq(lam, mu, c):
    a = lam / mu
    rho = a / c
    if rho >= 1:
        return np.nan
    # Erlang C
    s = sum(a**n / math.factorial(n) for n in range(c))
    last = a**c / (math.factorial(c) * (1 - rho))
    p0 = 1.0 / (s + last)
    pw = last * p0
    return pw / (c * mu - lam)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
mu = 1 / 4.0  # baseline service rate (E[S]=4h, estimated scenario value)
c = 12
Cs2 = 0.816**2  # squared CV from the estimated two-point mixture (s_fast=1.33h, s_slow=8h, p=0.60)
lam_grid = np.linspace(0.5, c * mu * 0.99, 100)
for cs2, ls, lab in zip([0.0, Cs2, 1.0], ["-", "--", ":"], ["det (C_S^2=0)", "mixture", "exp (C_S^2=1)"]):
    wq = [mmc_wq(l, mu, c) * (1 + cs2) / 2 if cs2 > 0 else mmc_wq(l, mu, c) for l in lam_grid]
    ax.plot(lam_grid, wq, color="0.2", ls=ls, lw=1.4, label=lab)
ax.set_xlabel("Arrival rate  $\\lambda$  (jobs/hour)")
ax.set_ylabel("Mean waiting time  $W_q$  (hours)")
ax.set_yscale("log")
ax.legend(title="Service-time variability", frameon=True, edgecolor="0.5")
savefig(fig, "fig5_mgc_waiting")
print("fig5 done")

fig5 done


## Tables - task parameters and residual-labour definitions

In [8]:
# ---- Tables ----
t1 = df[["task_id", "task_name", "pre_ai_hours", "post_ai_hours", "g_i",
         "verification_hours", "net_i", "becker", "accountability_constraint"]].copy()
t1.columns = ["Task", "Name", "Pre-AI (h)", "Post-AI (h)", "g_i (h)",
              "v_i (h)", "Net g-v (h)", "Delegation loss", "Accountability"]
t1.to_csv(os.path.join(TAB, "table1_task_parameters.csv"), index=False)
with open(os.path.join(TAB, "table1_task_parameters.tex"), "w") as fh:
    fh.write(t1.to_latex(index=False, escape=True,
             caption="Per-task delegation parameters.", label="tab:task"))

# Residual-labour share under alternative definitions
irr = df[df["compression_type"].isin(["irreducible", "partial_or_irreducible"])]
post_total = df["post_ai_hours"].sum()
defs = pd.DataFrame({
    "Definition": ["D1: irreducible / post-AI human time",
                   "D1': + half partial",
                   "D2: irreducible post / pre-AI total",
                   "D3: interview self-report",
                   "D4: task count"],
    "Value (%)": [
        round(100 * irr["post_ai_hours"].sum() / post_total, 1),
        round(100 * (irr["post_ai_hours"].sum() + 0.5 * df.loc[df.task_id=='T4','post_ai_hours'].iloc[0]) / post_total, 1),
        round(100 * irr["post_ai_hours"].sum() / df["pre_ai_hours"].sum(), 1),
        62.5,
        round(100 * len(irr) / len(df), 1),
    ]
})
defs.to_csv(os.path.join(TAB, "table2_residual_labour.csv"), index=False)

print("\nfigures:", sorted(os.listdir(FIG)))
print("tables:", sorted(os.listdir(TAB)))


figures: ['fig1_delegation_frontier.pdf', 'fig1_delegation_frontier.png', 'fig2_net_effect.pdf', 'fig2_net_effect.png', 'fig3_governance_capacity.pdf', 'fig3_governance_capacity.png', 'fig4_optimal_friction.pdf', 'fig4_optimal_friction.png', 'fig5_mgc_waiting.pdf', 'fig5_mgc_waiting.png']
tables: ['table1_task_parameters.csv', 'table1_task_parameters.tex', 'table2_residual_labour.csv']
